# Induction probe — pure recurrence vs hybrid, on the canonical task

The toy that motivated H-D3. Dense repeated-sequence induction (`[block | block]`,
predict the second half), scored on **solve-rate** and **steps-to-solve** over
multiple seeds — a single run is a coin flip (grokking), so it must be averaged.

Arms: `mingru` (0 attn), `whybrid` (windowed attn — bounded state, deployable),
`hybrid` (full attn — unbounded, reference only). Sweep `CB_S` across `2*window`
to find where the windowed arm falls back to MinGRU. No corpus/tokenizer needed.

In [ ]:
# --- setup: clone repo, deps, mount Drive (run once per session) ---
import os, subprocess, sys, time
if not os.path.exists('/content/CubbyLLM'):
    !git clone -q https://github.com/Grillcheese-AI/CubbyLLM.git /content/CubbyLLM
else:
    !cd /content/CubbyLLM && git pull -q --ff-only
!pip -q install torch numpy sentencepiece
from google.colab import drive; drive.mount('/content/drive')
REPO = '/content/CubbyLLM'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# --- run a script, tee to a log, catch a hung child on interrupt ---
def run(script, env_extra, log_name):
    env = dict(os.environ, CUBBY_SPM=TOKENIZER, CB_CORPUS=CORPUS, **env_extra)
    os.makedirs(f'{REPO}/validation/logs', exist_ok=True)
    log = f'{REPO}/validation/logs/{log_name}'
    p = None
    try:
        with open(log, 'a', encoding='utf-8', buffering=1) as f:
            f.write(f"\n=== {time.strftime('%F %T')} "
                    + ' '.join(f'{k}={v}' for k, v in sorted(env_extra.items())) + '\n')
            p = subprocess.Popen([sys.executable, '-u', f'validation/{script}'],
                                 cwd=REPO, env=env, stdout=subprocess.PIPE,
                                 stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in p.stdout:
                print(line, end=''); f.write(line)
            p.wait()
    finally:
        if p and p.poll() is None:      # never leave a child holding the GPU
            p.terminate()

In [ ]:
# --- one distance point: T = CB_S/2, window = CB_WINDOW ---
# within window (T < window) -> whybrid solves; beyond -> it collapses to mingru.
run('exp_d3_induction.py',
    dict(CB_S='128', CB_WINDOW='96', CB_SEEDS='4', CB_STEPS='8000',
         CB_D='128', CB_L='6', CB_ATTN_EVERY='3', CB_LR='3e-3', CB_LOG='500'),
    'induction_T64.log')

In [ ]:
# --- beyond window: T=256 > window=96 -> whybrid must collapse to mingru ---
run('exp_d3_induction.py',
    dict(CB_S='512', CB_WINDOW='96', CB_SEEDS='4', CB_STEPS='8000',
         CB_D='128', CB_L='6', CB_ATTN_EVERY='3', CB_LR='3e-3', CB_LOG='500'),
    'induction_T256.log')

### How to read

- **`solve rate`** — fraction of seeds reaching ≥90% second-half accuracy.
- **`median steps`** — how fast it groks (the discriminator when both solve).
- Within the window: `whybrid` and `hybrid` solve, `mingru` fails at distance.
- Beyond the window: `whybrid` collapses to `mingru` (a window can't reach past
  itself); only full `hybrid` still solves — the bounded-state limit, made visible.
- **HARNESS FAILURE** prints only if *nothing* trains (raise `CB_STEPS`/`CB_LR`).
  A run where `mingru`/`whybrid` solve and `attn` lags is normal — pure attention
  is the *hardest* arm here (it must learn the prev-token head from scratch).